In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from brainbox.task.trials import get_event_aligned_raster, get_psth
from iblatlas.atlas import AllenAtlas
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from brainbox.population.decode import get_spike_counts_in_bins
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox import singlecell
from tqdm.notebook import tqdm
import seaborn as sns

from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units

c:\Users\debot\micromamba\envs\info-decom-mirco\Lib\site-packages\scikits\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [2]:
df_granger = pd.read_csv("../data/processed/granger.csv")
df_decoding_region = pd.read_csv("../data/external/region_info.csv")
df_decoding_session_choice = pd.read_parquet("../data/external/choice_stage2.pqt")
df_decoding_session_stim = pd.read_parquet("../data/external/stimside_stage2.pqt")
df_decoding_session_feedback = pd.read_parquet("../data/external/feedback_stage2.pqt")

In [10]:
important_columns = [
    "Beryl",
    "Beryl.1",
    "# recordings",
    "# neurons",
    "# good neurons",
    "stim_dec",
    "stim_dec_sig",
    "prior_dec_ep",
    "prior_dec_ep_sig",
]

In [11]:
df = df_decoding_region[important_columns]

In [12]:
df_with_signficant_prior = df[df["prior_dec_ep_sig"] == True]

In [25]:
df_with_both = df_with_signficant_prior[df_with_signficant_prior["stim_dec_sig"] == True]

In [14]:
one = ONE(base_url="https://openalyx.internationalbrainlab.org", password="international")

Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"


In [15]:
bwm_df = bwm_query(one)
unit_df = bwm_units(one)

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01


In [23]:
# cross-ref regions, pids, eids and units; should be fast

In [26]:
# only keep regions of interest

regions_of_interest = df_with_both["Beryl"].unique()

In [46]:
units_regions_of_interest = unit_df[unit_df["Beryl"].isin(regions_of_interest)]

In [53]:
len(units_regions_of_interest["pid"].unique())

# need to download all these pids
# maybe download a fraction locally
# maybe all with ACAd

438

In [54]:
regions_of_interest

array(['MOs', 'SSp-ul', 'VISp', 'ACAd', 'PL', 'CP', 'VPM', 'MG', 'LGd',
       'ZI', 'SNr', 'MRN', 'SCm', 'PAG', 'APN', 'RN', 'PPN', 'PRNc',
       'PRNr', 'GRN', 'IRN', 'PGRN', 'CUL4 5', 'SIM', 'IP'], dtype=object)

In [57]:
eids = units_regions_of_interest["eid"].unique()

In [60]:
from ibl_info.utility import download_data

In [ ]:
for eid in eids:
    print(eid)
    pids, probes = one.eid2pid(eid)
    print(pids)
    for pid in pids:
        download_data(one, pid)
    break

In [64]:
pids[0][0]

UUID('1ab86a7f-578b-4a46-9c9c-df3be97abcca')